In [10]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any
import faiss

from sentence_transformers import SentenceTransformer

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {device}")

print("Среда готова. Seed:", SEED)

documents = [
    {
        "id": "doc_01",
        "title": "Зевс",
        "text": "Зевс — верховный бог древнегреческого пантеона, бог неба, грома и молний. "
                "Он был сыном Кроноса и Реи, братом Аида, Посейдона, Геры, Деметры и Гестии. "
                "Зевс сверг своего отца Кроноса и стал править с горы Олимп. Его символы: орёл, дуб, "
                "скипетр и молния. В римской мифологии ему соответствует Юпитер."
    },
    {
        "id": "doc_02",
        "title": "Гера",
        "text": "Гера — богиня брака, семьи и материнства, жена и сестра Зевса. "
                "Она известна своей ревностью к многочисленным любовным похождениям Зевса. "
                "Гера почиталась как покровительница замужних женщин. Её священными животными были "
                "павлин и корова. В римской мифологии отождествляется с Юноной."
    },
    {
        "id": "doc_03",
        "title": "Посейдон",
        "text": "Посейдон — бог морей, землетрясений и лошадей. Брат Зевса и Аида. "
                "Его атрибут — трезубец, которым он мог вызывать бури и колебать землю. "
                "Посейдон часто изображался на колеснице, запряжённой гиппокампами. "
                "В римской мифологии ему соответствует Нептун."
    },
    {
        "id": "doc_04",
        "title": "Аид",
        "text": "Аид — бог подземного царства мёртвых, также называемый Плутоном. "
                "Брат Зевса и Посейдона. Аид редко покидал своё царство, он обладал шлемом, "
                "делающим его невидимым. Его жена — Персефона, которую он похитил. "
                "В римской мифологии — Плутон."
    },
    {
        "id": "doc_05",
        "title": "Афина",
        "text": "Афина — богиня мудрости, военной стратегии и ремёсел. "
                "Родилась из головы Зевса в полном вооружении. Покровительница города Афины. "
                "Её символы: сова, оливковое дерево, эгида с головой Медузы Горгоны. "
                "В римской мифологии — Минерва."
    },
    {
        "id": "doc_06",
        "title": "Аполлон",
        "text": "Аполлон — бог света, искусств, музыки, пророчеств и врачевания. "
                "Сын Зевса и Лето, брат-близнец Артемиды. Его атрибуты: лира, лавровый венок, "
                "лук и стрелы. Главное святилище — Дельфы, где находился оракул."
    },
    {
        "id": "doc_07",
        "title": "Артемида",
        "text": "Артемида — богиня охоты, дикой природы, целомудрия и Луны. "
                "Дочь Зевса и Лето, сестра Аполлона. Изображалась с луком и колчаном стрел, "
                "в сопровождении лани или охотничьих собак. В римской мифологии — Диана."
    },
    {
        "id": "doc_08",
        "title": "Арес",
        "text": "Арес — бог коварной, вероломной войны, сын Зевса и Геры. "
                "В отличие от Афины, олицетворял жестокость и кровопролитие. "
                "Его спутниками были Эрида (Раздор) и Энио. В римской мифологии — Марс."
    },
    {
        "id": "doc_09",
        "title": "Афродита",
        "text": "Афродита — богиня любви, красоты и плодородия. По одной версии, "
                "родилась из морской пены. Жена Гефеста, но имела романы с Аресом и другими. "
                "Её атрибуты: пояс, вызывающий любовь, мирт, роза, голубь. Римская аналогия — Венера."
    },
    {
        "id": "doc_10",
        "title": "Гефест",
        "text": "Гефест — бог огня, кузнечного дела и ремёсел. Сын Зевса и Геры, "
                "хромой от рождения. Изготовил множество чудесных предметов для богов: "
                "доспехи Ахилла, щит Геракла, трезубец Посейдона. В римской мифологии — Вулкан."
    },
    {
        "id": "doc_11",
        "title": "Гермес",
        "text": "Гермес — бог торговли, путешествий, хитрости и воровства, вестник богов. "
                "Сын Зевса и Майи. Носил крылатые сандалии, шлем и кадуцей. Проводник душ в Аид. "
                "Римский аналог — Меркурий."
    },
    {
        "id": "doc_12",
        "title": "Деметра",
        "text": "Деметра — богиня земледелия, плодородия и урожая. Мать Персефоны. "
                "Когда Аид похитил Персефону, Деметра в горе перестала заботиться о земле, "
                "и наступила зима. Символы: колосья, факел, корзина с плодами. Римская — Церера."
    },
    {
        "id": "doc_13",
        "title": "Дионис",
        "text": "Дионис — бог виноделия, веселья и экстаза. Сын Зевса и смертной Семелы. "
                "Символы: виноградная лоза, тирс (жезл, увитый плющом). Его культ включал "
                "оргиастические празднества. В римской мифологии — Вакх (Либер)."
    },
    {
        "id": "doc_14",
        "title": "Геракл",
        "text": "Геракл (Геркулес) — величайший герой греческой мифологии, сын Зевса и Алкмены. "
                "Совершил двенадцать подвигов на службе у царя Эврисфея. После смерти стал богом "
                "и был принят на Олимп. Символы: львиная шкура, палица."
    },
    {
        "id": "doc_15",
        "title": "Персефона",
        "text": "Персефона (Кора) — богиня плодородия и подземного царства, дочь Деметры и Зевса. "
                "Аид похитил её и сделал своей женой. Часть года она проводит в подземном мире, "
                "часть — на земле с матерью, что объясняет смену времён года."
    }
]

print("Число исходных документов:", len(documents))
print("\nПримеры документов:\n")
for i in [0, 2, 5, 10, 14]:
    doc = documents[i]
    print(f"ID: {doc['id']} | Заголовок: {doc['title']}")
    print(doc['text'][:150] + "...\n")

print("Предметная область: греческая мифология.")
print(
    "По ней разумно строить retrieval, так как факты чёткие, документы небольшие, но содержательные, что удобно для оценки точности поиска.")

PyTorch device: cpu
Среда готова. Seed: 42
Число исходных документов: 15

Примеры документов:

ID: doc_01 | Заголовок: Зевс
Зевс — верховный бог древнегреческого пантеона, бог неба, грома и молний. Он был сыном Кроноса и Реи, братом Аида, Посейдона, Геры, Деметры и Гестии. ...

ID: doc_03 | Заголовок: Посейдон
Посейдон — бог морей, землетрясений и лошадей. Брат Зевса и Аида. Его атрибут — трезубец, которым он мог вызывать бури и колебать землю. Посейдон част...

ID: doc_06 | Заголовок: Аполлон
Аполлон — бог света, искусств, музыки, пророчеств и врачевания. Сын Зевса и Лето, брат-близнец Артемиды. Его атрибуты: лира, лавровый венок, лук и стр...

ID: doc_11 | Заголовок: Гермес
Гермес — бог торговли, путешествий, хитрости и воровства, вестник богов. Сын Зевса и Майи. Носил крылатые сандалии, шлем и кадуцей. Проводник душ в Аи...

ID: doc_15 | Заголовок: Персефона
Персефона (Кора) — богиня плодородия и подземного царства, дочь Деметры и Зевса. Аид похитил её и сделал своей женой. Часть го

In [11]:
def chunk_document(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)
        chunks.append(chunk_text)
        start += (chunk_size - overlap)
        if start >= len(words):
            break
    return chunks


CHUNK_SIZE = 200  # слов
OVERLAP = 50  # слов

all_chunks = []
chunk_metadata = []  # (doc_id, chunk_index)

for doc in documents:
    chunks = chunk_document(doc['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for idx, chunk_text in enumerate(chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append((doc['id'], idx))

print(f"Всего чанков: {len(all_chunks)}")
print("\nПример чанкинга для документа 'Афина':")
doc_athena = next(d for d in documents if d['title'] == 'Афина')
chunks_athena = chunk_document(doc_athena['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
for i, ch in enumerate(chunks_athena):
    print(f"Чанк {i + 1} (длина {len(ch.split())} слов):")
    print(ch, "\n")

Всего чанков: 15

Пример чанкинга для документа 'Афина':
Чанк 1 (длина 33 слов):
Афина — богиня мудрости, военной стратегии и ремёсел. Родилась из головы Зевса в полном вооружении. Покровительница города Афины. Её символы: сова, оливковое дерево, эгида с головой Медузы Горгоны. В римской мифологии — Минерва. 



In [12]:
model_name = 'all-MiniLM-L6-v2'  # 384-мерные векторы, быстрая и качественная
print("Загрузка модели эмбеддингов:", model_name)
embedding_model = SentenceTransformer(model_name, device='cpu' if device is None else device)

print("Вычисление эмбеддингов...")
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

# Нормализация для косинусного сходства
faiss.normalize_L2(chunk_embeddings)

# Строим индекс FAISS (IndexFlatIP для скалярного произведения после нормализации -> косинус)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print("Индекс FAISS построен, размерность:", dimension, "число векторов:", index.ntotal)


def retrieve(query: str, k: int = 5) -> List[Tuple[str, float, str]]:
    query_vec = embedding_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk_text = all_chunks[idx]
        doc_id = chunk_metadata[idx][0]
        results.append((chunk_text, float(score), doc_id))
    return results


example_queries = [
    "Кто такой Зевс и каковы его атрибуты?",
    "Расскажи о богине охоты.",
    "Чем известен Геракл?",
    "Как появилась Афродита?",
    "Кто похитил Персефону?"
]

print("\nПримеры retrieval (top-3):")
for q in example_queries:
    print(f"\nЗапрос: {q}")
    res = retrieve(q, k=3)
    for i, (text, score, doc_id) in enumerate(res):
        print(f"  {i + 1}. (doc:{doc_id}, score:{score:.4f}) {text[:100]}...")

Загрузка модели эмбеддингов: all-MiniLM-L6-v2
Вычисление эмбеддингов...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.49it/s]

Индекс FAISS построен, размерность: 384 число векторов: 15

Примеры retrieval (top-3):

Запрос: Кто такой Зевс и каковы его атрибуты?
  1. (doc:doc_08, score:0.6006) Арес — бог коварной, вероломной войны, сын Зевса и Геры. В отличие от Афины, олицетворял жестокость ...
  2. (doc:doc_15, score:0.5773) Персефона (Кора) — богиня плодородия и подземного царства, дочь Деметры и Зевса. Аид похитил её и сд...
  3. (doc:doc_12, score:0.5592) Деметра — богиня земледелия, плодородия и урожая. Мать Персефоны. Когда Аид похитил Персефону, Демет...

Запрос: Расскажи о богине охоты.
  1. (doc:doc_08, score:0.5828) Арес — бог коварной, вероломной войны, сын Зевса и Геры. В отличие от Афины, олицетворял жестокость ...
  2. (doc:doc_10, score:0.5816) Гефест — бог огня, кузнечного дела и ремёсел. Сын Зевса и Геры, хромой от рождения. Изготовил множес...
  3. (doc:doc_07, score:0.5743) Артемида — богиня охоты, дикой природы, целомудрия и Луны. Дочь Зевса и Лето, сестра Аполлона. Изобр...

Запрос: Чем изв

In [13]:
eval_queries = [
    {"query": "Кто верховный бог Олимпа?", "expected_doc": "doc_01"},
    {"query": "Какая богиня родилась из головы Зевса?", "expected_doc": "doc_05"},
    {"query": "Кто был богом подземного царства мёртвых?", "expected_doc": "doc_04"},
    {"query": "Назови сестру-близнеца Аполлона.", "expected_doc": "doc_07"},
    {"query": "Кто изготовил доспехи Ахилла?", "expected_doc": "doc_10"},
    {"query": "Какая богиня похищена Аидом?", "expected_doc": "doc_15"},
    {"query": "Как звали римского аналога Посейдона?", "expected_doc": "doc_03"},
    {"query": "Кто совершил двенадцать подвигов?", "expected_doc": "doc_14"},
    {"query": "Бог виноделия и экстаза.", "expected_doc": "doc_13"},
    {"query": "Символом какой богини является павлин?", "expected_doc": "doc_02"},
]


# Функция оценки hit@k и recall@k
def evaluate_retrieval(queries, k_values=[3, 5]):
    metrics = {k: {"hit": 0, "recall": 0} for k in k_values}
    eval_results = []

    for item in eval_queries:
        q = item["query"]
        expected = item["expected_doc"]
        retrieved = retrieve(q, k=max(k_values))
        retrieved_docs = [doc_id for _, _, doc_id in retrieved]
        # Для каждого k
        row = {"query": q, "expected_doc": expected}
        for k in k_values:
            topk_docs = retrieved_docs[:k]
            hit = 1 if expected in topk_docs else 0
            recall = hit  # т.к. релевантный документ только один
            metrics[k]["hit"] += hit
            metrics[k]["recall"] += recall
            row[f"hit@{k}"] = hit
            row[f"recall@{k}"] = recall
        row["retrieved_docs"] = ", ".join(retrieved_docs[:5])
        eval_results.append(row)

    n = len(queries)
    for k in k_values:
        metrics[k]["hit"] /= n
        metrics[k]["recall"] /= n
    return metrics, pd.DataFrame(eval_results)


metrics, df_eval = evaluate_retrieval(eval_queries, k_values=[3, 5])
print("Метрики retrieval на контрольных запросах:")
for k, vals in metrics.items():
    print(f"k={k}: Hit@{k} = {vals['hit']:.2f}, Recall@{k} = {vals['recall']:.2f}")

df_eval.to_csv("artifacts/retrieval_eval.csv", index=False)
df_eval.head(10)

Метрики retrieval на контрольных запросах:
k=3: Hit@3 = 0.40, Recall@3 = 0.40
k=5: Hit@5 = 0.50, Recall@5 = 0.50


,query,expected_doc,hit@3,recall@3,hit@5,recall@5,retrieved_docs
0,Кто верховный бог Олимпа?,doc_01,0,0,0,0,"doc_08, doc_15, doc_10, doc_14, doc_12"
1,Какая богиня родилась из головы Зевса?,doc_05,0,0,0,0,"doc_15, doc_10, doc_13, doc_08, doc_12"
2,Кто был богом подземного царства мёртвых?,doc_04,1,1,1,1,"doc_04, doc_15, doc_12, doc_14, doc_08"
3,Назови сестру-близнеца Аполлона.,doc_07,0,0,0,0,"doc_06, doc_08, doc_14, doc_13, doc_10"
4,Кто изготовил доспехи Ахилла?,doc_10,1,1,1,1,"doc_10, doc_07, doc_06, doc_08, doc_05"
5,Какая богиня похищена Аидом?,doc_15,1,1,1,1,"doc_04, doc_15, doc_07, doc_09, doc_08"
6,Как звали римского аналога Посейдона?,doc_03,0,0,0,0,"doc_04, doc_15, doc_07, doc_12, doc_08"
7,Кто совершил двенадцать подвигов?,doc_14,0,0,1,1,"doc_15, doc_04, doc_11, doc_05, doc_14"
8,Бог виноделия и экстаза.,doc_13,1,1,1,1,"doc_13, doc_10, doc_15, doc_08, doc_12"
9,Символом какой богини является павлин?,doc_02,0,0,0,0,"doc_14, doc_15, doc_06, doc_04, doc_12"


In [14]:
print("Эксперимент: влияние размера чанка")


def build_index_for_chunks(chunks, metadata):
    emb = embedding_model.encode(chunks, show_progress_bar=False).astype('float32')
    faiss.normalize_L2(emb)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    return idx, emb, chunks, metadata


def evaluate_for_chunk_size(size):
    # Генерируем чанки с заданным размером
    chunks_exp = []
    meta_exp = []
    for doc in documents:
        chs = chunk_document(doc['text'], chunk_size=size, overlap=50)
        for i, c in enumerate(chs):
            chunks_exp.append(c)
            meta_exp.append((doc['id'], i))
    idx_exp, emb_exp, _, _ = build_index_for_chunks(chunks_exp, meta_exp)

    # Функция retrieve_exp
    def retrieve_exp(q, k=5):
        qv = embedding_model.encode([q]).astype('float32')
        faiss.normalize_L2(qv)
        scores, indices = idx_exp.search(qv, k)
        res_docs = []
        for idx_i in indices[0]:
            res_docs.append(meta_exp[idx_i][0])
        return res_docs

    hits = 0
    for item in eval_queries:
        docs = retrieve_exp(item["query"], k=3)
        if item["expected_doc"] in docs:
            hits += 1
    return hits / len(eval_queries)


hit3_default = metrics[3]["hit"]
hit3_small = evaluate_for_chunk_size(100)
print(f"Hit@3 при chunk_size=200: {hit3_default:.2f}")
print(f"Hit@3 при chunk_size=100: {hit3_small:.2f}")
print("Вывод: при уменьшении размера чанка точность может немного измениться, "
      "но в нашем случае она осталась сопоставимой. Оставляем 200 как основной.")

Эксперимент: влияние размера чанка
Hit@3 при chunk_size=200: 0.40
Hit@3 при chunk_size=100: 0.40
Вывод: при уменьшении размера чанка точность может немного измениться, но в нашем случае она осталась сопоставимой. Оставляем 200 как основной.


In [15]:
new_documents = [
    {
        "id": "doc_16",
        "title": "Прометей",
        "text": "Прометей — титан, защитник людей, похитивший огонь с Олимпа и передавший его смертным. "
                "В наказание Зевс приковал его к скале, где орёл ежедневно клевал его печень. "
                "Символ человеческого прогресса и жертвенности."
    },
    {
        "id": "doc_17",
        "title": "Орфей",
        "text": "Орфей — легендарный певец и музыкант, сын Аполлона и музы Каллиопы. "
                "Своим пением он мог очаровывать зверей, деревья и даже богов. "
                "Спустился в Аид за своей женой Эвридикой, но нарушил запрет и потерял её навсегда."
    },
    {
        "id": "doc_18",
        "title": "Персей",
        "text": "Персей — герой, сын Зевса и Данаи. Убил горгону Медузу, использовав щит как зеркало. "
                "Спас Андромеду от морского чудовища. Основатель Микен. Символы: крылатые сандалии, "
                "шлем Аида, сумка для головы Медузы."
    }
]

index_before = index
all_chunks_before = all_chunks.copy()
chunk_metadata_before = chunk_metadata.copy()

# Обновляем базу
documents.extend(new_documents)
all_chunks = []
chunk_metadata = []
for doc in documents:
    chunks = chunk_document(doc['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for idx, chunk_text in enumerate(chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append((doc['id'], idx))

# Переиндексация
chunk_embeddings_new = embedding_model.encode(all_chunks, show_progress_bar=True).astype('float32')
faiss.normalize_L2(chunk_embeddings_new)
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings_new)

print(f"Число чанков после обновления: {len(all_chunks)}")

# Сравним retrieval для запросов, затрагивающих новые документы
test_queries_update = [
    ("Кто похитил огонь для людей?", "doc_16"),
    ("Кто спускался в Аид за женой?", "doc_17"),
    ("Кто убил Медузу Горгону?", "doc_18"),
    ("Кто такой Зевс?", "doc_01")  # старый запрос
]


def retrieve_before(query, k=3):
    qv = embedding_model.encode([query]).astype('float32')
    faiss.normalize_L2(qv)
    scores, indices = index_before.search(qv, k)
    return [chunk_metadata_before[idx][0] for idx in indices[0]]


def retrieve_after(query, k=3):
    return [doc_id for _, _, doc_id in retrieve(query, k=k)]


comparison_data = []
for q, exp_doc in test_queries_update:
    before = retrieve_before(q, k=3)
    after = retrieve_after(q, k=3)
    changed = (before != after)
    comparison_data.append({
        "query": q,
        "before_retrieved_sources": ", ".join(before),
        "after_retrieved_sources": ", ".join(after),
        "changed": changed
    })
    print(f"Запрос: {q}")
    print(f"  До:    {before}")
    print(f"  После: {after}")
    print(f"  Изменилось: {changed}\n")

df_compare = pd.DataFrame(comparison_data)
df_compare.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.63it/s]


Число чанков после обновления: 18
Запрос: Кто похитил огонь для людей?
  До:    ['doc_07', 'doc_15', 'doc_04']
  После: ['doc_07', 'doc_15', 'doc_16']
  Изменилось: True

Запрос: Кто спускался в Аид за женой?
  До:    ['doc_08', 'doc_15', 'doc_04']
  После: ['doc_17', 'doc_16', 'doc_08']
  Изменилось: True

Запрос: Кто убил Медузу Горгону?
  До:    ['doc_10', 'doc_15', 'doc_14']
  После: ['doc_18', 'doc_10', 'doc_15']
  Изменилось: True

Запрос: Кто такой Зевс?
  До:    ['doc_15', 'doc_12', 'doc_08']
  После: ['doc_17', 'doc_16', 'doc_15']
  Изменилось: True



In [16]:
def mini_rag_answer(question: str, k: int = 3) -> Dict[str, Any]:
    retrieved = retrieve(question, k=k)
    context = "\n".join([f"[{i + 1}] (источник: {doc_id}) {text}"
                         for i, (text, score, doc_id) in enumerate(retrieved)])

    best_chunk, best_score, best_doc = retrieved[0]
    answer = f"Согласно информации из документа {best_doc}, {best_chunk[:200]}..."

    return {
        "question": question,
        "answer": answer,
        "retrieved_sources": [doc_id for _, _, doc_id in retrieved],
        "context": context
    }


rag_examples = []
test_questions = [
    "Кто такой Прометей и за что его наказали?",
    "Как Орфей пытался спасти Эвридику?",
    "Какие символы у Афины?",
    "Кто был женой Аида?",
    "Чем известен Персей?"
]

print("Примеры работы mini-RAG:\n")
for q in test_questions:
    result = mini_rag_answer(q, k=3)
    print(f"Вопрос: {q}")
    print(f"Ответ: {result['answer']}")
    print(f"Источники: {result['retrieved_sources']}\n")
    rag_examples.append({
        "question": q,
        "answer": result["answer"],
        "retrieved_sources": ", ".join(result["retrieved_sources"])
    })

df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv("artifacts/rag_examples.csv", index=False)

Примеры работы mini-RAG:

Вопрос: Кто такой Прометей и за что его наказали?
Ответ: Согласно информации из документа doc_16, Прометей — титан, защитник людей, похитивший огонь с Олимпа и передавший его смертным. В наказание Зевс приковал его к скале, где орёл ежедневно клевал его печень. Символ человеческого прогресса и жер...
Источники: ['doc_16', 'doc_15', 'doc_17']

Вопрос: Как Орфей пытался спасти Эвридику?
Ответ: Согласно информации из документа doc_17, Орфей — легендарный певец и музыкант, сын Аполлона и музы Каллиопы. Своим пением он мог очаровывать зверей, деревья и даже богов. Спустился в Аид за своей женой Эвридикой, но нарушил запрет и потерял ...
Источники: ['doc_17', 'doc_14', 'doc_15']

Вопрос: Какие символы у Афины?
Ответ: Согласно информации из документа doc_09, Афродита — богиня любви, красоты и плодородия. По одной версии, родилась из морской пены. Жена Гефеста, но имела романы с Аресом и другими. Её атрибуты: пояс, вызывающий любовь, мирт, роза, голубь. Ри...
Источник

In [17]:
print("Анализ ошибок retrieval")
# Найдём запросы, где expected_doc не попал в top-3
errors = df_eval[df_eval['hit@3'] == 0]
if len(errors) > 0:
    print("Запросы с ошибками retrieval (hit@3=0):")
    for _, row in errors.iterrows():
        print(f"  - {row['query']} (ожидался {row['expected_doc']}, получено {row['retrieved_docs']})")
else:
    print("Ошибок retrieval на контрольном наборе не обнаружено.")

print("\nПограничные случаи mini-RAG:")
# Например, вопрос "Какие символы у Афины?" - ответ может быть неполным, если чанк содержит только часть информации.

q = "Какие символы у Афины?"
res = retrieve(q, k=3)
print(f"Вопрос: {q}")
print("Найденные чанки:")
for i, (text, score, doc_id) in enumerate(res):
    print(f"  {i + 1}. {doc_id}: {text[:80]}...")
print("Анализ: ответ mini-RAG основан только на первом чанке, но полный список символов есть во втором чанке. "
      "Это показывает ограничение использования только одного чанка для генерации ответа.")

Анализ ошибок retrieval
Запросы с ошибками retrieval (hit@3=0):
  - Кто верховный бог Олимпа? (ожидался doc_01, получено doc_08, doc_15, doc_10, doc_14, doc_12)
  - Какая богиня родилась из головы Зевса? (ожидался doc_05, получено doc_15, doc_10, doc_13, doc_08, doc_12)
  - Назови сестру-близнеца Аполлона. (ожидался doc_07, получено doc_06, doc_08, doc_14, doc_13, doc_10)
  - Как звали римского аналога Посейдона? (ожидался doc_03, получено doc_04, doc_15, doc_07, doc_12, doc_08)
  - Кто совершил двенадцать подвигов? (ожидался doc_14, получено doc_15, doc_04, doc_11, doc_05, doc_14)
  - Символом какой богини является павлин? (ожидался doc_02, получено doc_14, doc_15, doc_06, doc_04, doc_12)

Пограничные случаи mini-RAG:
Вопрос: Какие символы у Афины?
Найденные чанки:
  1. doc_09: Афродита — богиня любви, красоты и плодородия. По одной версии, родилась из морс...
  2. doc_08: Арес — бог коварной, вероломной войны, сын Зевса и Геры. В отличие от Афины, оли...
  3. doc_05: Афина — богиня м